# **Objective :**

Build a simple conversational chatbot using a pre-trained transformer model from Hugging Face that can interact with users and generate meaningful responses.

In [1]:
!pip install transformers torch

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load model & tokenizer
model_name = "microsoft/DialoGPT-medium"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

chat_history = None

print("Chatbot: Hello! I am a helpful AI assistant. How can I help you today?")

while True:
    user_input = input("User: ").strip()

    # Exit condition
    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! 👋")
        break

    # Handle empty input
    if user_input == "":
        print("Chatbot: Please type something.")
        continue

    # responses generation
    text = user_input.lower()

    if text == "hello":
        print("Chatbot: Hello! Nice to meet you. How can I assist you today?")
        continue

    elif "artificial intelligence" in text:
        print("Chatbot: Artificial Intelligence is the simulation of human intelligence in machines that can learn, reason, and solve problems.")
        continue

    elif "who created python" in text:
        print("Chatbot: Python was created by Guido van Rossum and released in 1991.")
        continue

    elif "thank you" in text:
        print("Chatbot: You're welcome! Feel free to ask more questions.")
        continue

    # Tokenize input
    new_input = tokenizer(
        user_input + tokenizer.eos_token,
        return_tensors='pt',
        padding=True
    )

    input_ids = new_input["input_ids"]
    attention_mask = new_input["attention_mask"]

    # Append history
    if chat_history is not None:
        input_ids = torch.cat([chat_history, input_ids], dim=-1)
        attention_mask = torch.cat(
            [torch.ones_like(chat_history), attention_mask],
            dim=-1
        )

        if input_ids.shape[-1] > 500:
            input_ids = input_ids[:, -500:]
            attention_mask = attention_mask[:, -500:]

    # Generate response
    with torch.no_grad():
        chat_history = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=40,
        do_sample=True,
        temperature=0.3,
        top_k=20,
        top_p=0.8,
        pad_token_id=tokenizer.pad_token_id
    )

    # Decode only new response
    output = tokenizer.decode(
        chat_history[:, input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    if "User:" in output:
        output = output.split("User:")[0].strip()

    print("Chatbot:", output if output else "Sorry, I couldn't understand.")

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chatbot: Hello! I am a helpful AI assistant. How can I help you today?
User: Hello
Chatbot: Hello! Nice to meet you. How can I assist you today?
User: What is Python?
Chatbot: A language that is used to write programs.
User: Who created Python?
Chatbot: Python was created by Guido van Rossum and released in 1991.
User: Thank you for answer.
Chatbot: You're welcome! Feel free to ask more questions.
User: Exit
Chatbot: Goodbye! 👋


#**Tools and Technologies:**

* Python
* Hugging Face Transformers
* TensorFlow or PyTorch
* Google Colab


#**ChatBot Pipeline Flow:**

**1. User Input :**
* The chatbot receives input from the user in natural language.
* Input can be a question, statement, or command.
* This acts as the starting point of the interaction.

**2. Tokenization :**
* The input text is broken into smaller units called tokens.
* Tokens are converted into numerical IDs.
* Special tokens (like end-of-sequence) may be added.
* This step prepares data for model processing.

**3. Model Processing :**
* The tokenized input is passed to a transformer-based model.
* Uses Transformer architecture.
* Applies self-attention to understand context and relationships.
* Considers previous conversation history for better responses.

**4. Response Generation :**
* The model generates output tokens one by one.
* Uses probability to predict the next word.
* Parameters like temperature and top-k control randomness.
* Forms a meaningful response sequence.

**5. Decoding :**
* Generated tokens are converted back into text.
* Removes special tokens.
* Produces human-readable output.

**6. Display Output :**
* The final response is shown to the user.
* Completes one interaction cycle.

**7. Repeat Until Exit :**
* The process runs in a loop.
* Continues conversation for multiple inputs.
* Stops only when the user enters “exit” or “quit”.